# 🔐 Login Gerektiren Siteleri Scraping

Bu notebook'ta **en zorlu scraping konularından** birini öğreniyoruz!

## 🎯 Bu Derste Öğrenecekleriniz:
1. **Session Management** - Oturum yönetimi
2. **Cookie Handling** - Çerez yönetimi  
3. **Form Data** gönderme
4. **CSRF Token** yönetimi
5. **Headers** ve **Referrer** ayarlama
6. **Multi-step Authentication** - Çok adımlı doğrulama

---

## ⚠️ ETİK UYARI
Bu teknikler **sadece eğitim amaçlı** ve **izniniz olan siteler** için kullanılmalıdır!

### ✅ İzin verilen durumlar:
- Kendi web siteniz
- Test siteleri (httpbin, quotes.toscrape)
- API'nin olmadığı ve ToS'a uygun durumlar

### ❌ Yasak durumlar:
- Başkalarının hesaplarına izinsiz erişim
- Sosyal medya platformları (ToS ihlali)
- Bankacılık ve finans siteleri
- Kişisel veri içeren platformlar

In [1]:
# Gerekli kütüphaneleri yükleyelim
import requests
from bs4 import BeautifulSoup
import time
import json
from urllib.parse import urljoin, urlparse
import re
from pprint import pprint

print("✅ Kütüphaneler yüklendi!")
print("🔐 Login scraping dersine hazırız!")

✅ Kütüphaneler yüklendi!
🔐 Login scraping dersine hazırız!


---

## 🍪 Örnek 1: Session ve Cookie Temelleri

**Login scraping'in kalbi: Session Management**

### 🤔 Neden Session Gerekli?
- HTTP **stateless** (durumsuz) bir protokol
- Server sizi **tanımaz** (her istek yeni)
- **Cookie'ler** ile kimliğinizi hatırlar
- **Session** ile login durumunuz korunur

### 📚 Session vs Normal Request Farkı:

In [2]:
print("🔍 NORMAL REQUEST vs SESSION KARŞILAŞTIRMASI")
print("=" * 60)

# Test URL'i
test_url = "https://httpbin.org/cookies"

# 1. Normal Request - Cookie'ler korunmaz
print("\n1️⃣ NORMAL REQUEST (Cookie'siz):")
print("-" * 40)

response1 = requests.get(test_url)
print(f"📊 Status: {response1.status_code}")
print(f"🍪 Cookie'ler: {response1.json()}")

# 2. Session ile - Cookie'ler korunur
print("\n2️⃣ SESSION İLE (Cookie korumalı):")
print("-" * 40)

# Session oluştur
session = requests.Session()

# Önce cookie set edelim
session.get("https://httpbin.org/cookies/set/test_cookie/hello_world")

# Şimdi cookie'leri kontrol edelim
response2 = session.get(test_url)
print(f"📊 Status: {response2.status_code}")
print(f"🍪 Cookie'ler: {response2.json()}")

print("\n💡 Fark gördünüz mü? Session cookie'leri hatırlıyor!")

🔍 NORMAL REQUEST vs SESSION KARŞILAŞTIRMASI

1️⃣ NORMAL REQUEST (Cookie'siz):
----------------------------------------
📊 Status: 200
🍪 Cookie'ler: {'cookies': {}}

2️⃣ SESSION İLE (Cookie korumalı):
----------------------------------------
📊 Status: 200
🍪 Cookie'ler: {'cookies': {'test_cookie': 'hello_world'}}

💡 Fark gördünüz mü? Session cookie'leri hatırlıyor!


### 🛠️ Session Nesnesinin Avantajları

| Özellik | Normal Request | Session |
|---------|---------------|---------|
| **Cookie Saklama** | ❌ Her istek yeni | ✅ Otomatik saklar |
| **Headers Korunma** | ❌ Her seferinde set | ✅ Bir kez set, hep geçerli |
| **Connection Pooling** | ❌ Her istek yeni bağlantı | ✅ Bağlantıları yeniden kullanır |
| **Performance** | 🐌 Yavaş | 🚀 Hızlı |
| **Login State** | ❌ Korunamaz | ✅ Korunur |

---

## 📝 Örnek 2: CSRF Token Yönetimi

**Gerçek siteler genelde CSRF koruması kullanır!**

### 🤔 CSRF Token Nedir?
- **Cross-Site Request Forgery** koruması
- **Hidden form field** olarak gelir
- **Her session** için farklı değer
- **POST ile birlikte** gönderilmeli

In [3]:
def find_csrf_token(soup):
    """
    HTML'den CSRF token'ını bulmak için farklı yöntemler dener
    """
    csrf_patterns = [
        # Yaygın CSRF field isimleri
        ('name', 'csrf_token'),
        ('name', 'csrfmiddlewaretoken'),
        ('name', '_token'),
        ('name', 'authenticity_token'),
        ('name', '__RequestVerificationToken'),
        
        # Meta tag'lerde CSRF
        ('name', 'csrf-token'),
        ('property', 'csrf-token'),
    ]
    
    print("🔍 CSRF token aranıyor...")
    
    # Input field'larında ara
    for attr_name, attr_value in csrf_patterns:
        element = soup.find('input', {attr_name: attr_value})
        if element:
            token = element.get('value')
            if token:
                print(f"✅ CSRF token bulundu! Field: {attr_value}")
                print(f"   Token: {token[:20]}...")
                return attr_value, token
    
    # Meta tag'lerde ara
    for attr_name, attr_value in csrf_patterns:
        element = soup.find('meta', {attr_name: attr_value})
        if element:
            token = element.get('content')
            if token:
                print(f"✅ CSRF token meta tag'de bulundu! Field: {attr_value}")
                print(f"   Token: {token[:20]}...")
                return attr_value, token
    
    print("⚠️ CSRF token bulunamadı")
    return None, None

# Test HTML'i (CSRF token içeren örnek)
test_html = """
<html>
<body>
    <form method="post" action="/login">
        <input type="hidden" name="csrf_token" value="abc123def456ghi789">
        <input type="text" name="username" placeholder="Username">
        <input type="password" name="password" placeholder="Password">
        <button type="submit">Login</button>
    </form>
</body>
</html>
"""

test_soup = BeautifulSoup(test_html, 'html.parser')
csrf_field, csrf_token = find_csrf_token(test_soup)

if csrf_token:
    print(f"\n🎯 Bulunan CSRF bilgileri:")
    print(f"   Field adı: {csrf_field}")
    print(f"   Token değeri: {csrf_token}")

print("\n✅ CSRF token bulma fonksiyonu hazırlandı!")

🔍 CSRF token aranıyor...
✅ CSRF token bulundu! Field: csrf_token
   Token: abc123def456ghi789...

🎯 Bulunan CSRF bilgileri:
   Field adı: csrf_token
   Token değeri: abc123def456ghi789

✅ CSRF token bulma fonksiyonu hazırlandı!


---

## 🔄 Örnek 3: Tam Login Workflow

**Gerçek dünya örneği: Quotes to Scrape Login**

Bu örnekte tüm öğrendiklerimizi birleştiriyoruz:
- Session yönetimi
- CSRF token
- Form data
- Login kontrolü
- Korumalı sayfa erişimi

In [4]:
# Quotes to Scrape sitesinin login özelliğini kullanalım
print("🎯 TAM LOGIN WORKFLOW ÖRNEĞİ")
print("=" * 50)

# Quotes to Scrape login sistemi
base_url = "http://quotes.toscrape.com"
login_url = f"{base_url}/login"
protected_url = f"{base_url}/"  # Ana sayfa (login sonrası farklı görünecek)

# Test kullanıcı bilgileri (Bu site için geçerli)
test_username = "admin"
test_password = "admin"

print(f"🌐 Base URL: {base_url}")
print(f"🔐 Login URL: {login_url}")
print(f"👤 Test kullanıcı: {test_username}")

# Session oluştur
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Educational Bot; Quotes Login Practice)'
})

print("\n🚀 Login süreci başlatılıyor...")

🎯 TAM LOGIN WORKFLOW ÖRNEĞİ
🌐 Base URL: http://quotes.toscrape.com
🔐 Login URL: http://quotes.toscrape.com/login
👤 Test kullanıcı: admin

🚀 Login süreci başlatılıyor...


In [5]:
# ADIM 1: Login sayfasını ziyaret et
print("\n1️⃣ Login sayfası getiriliyor...")
login_page = session.get(login_url)

if login_page.status_code == 200:
    print(f"✅ Login sayfası alındı: {len(login_page.text)} karakter")
    
    # ADIM 2: CSRF token ve form analizi
    print("\n2️⃣ Form analiz ediliyor...")
    soup = BeautifulSoup(login_page.text, 'html.parser')
    
    # Login form'unu bul
    login_form = soup.find('form')
    
    if login_form:
        print(f"📋 Login form bulundu!")
        
        # Form field'larını analiz et
        fields = login_form.find_all('input')
        print(f"📝 Form field'ları ({len(fields)} adet):")
        
        form_data = {}
        for field in fields:
            name = field.get('name', '')
            field_type = field.get('type', 'text')
            value = field.get('value', '')
            
            if name:
                print(f"   🔸 {name} ({field_type}): '{value}'")
                
                # Hidden field'ları otomatik ekle
                if field_type == 'hidden':
                    form_data[name] = value
        
        # CSRF token kontrolü
        csrf_field, csrf_token = find_csrf_token(soup)
        if csrf_token:
            form_data[csrf_field] = csrf_token
            print(f"🛡️ CSRF token eklendi: {csrf_field}")
        
        # ADIM 3: Login bilgilerini ekle ve gönder
        print("\n3️⃣ Login bilgileri hazırlanıyor...")
        
        # Username ve password field'larını bul
        username_field = 'username'  # Quotes.toscrape için
        password_field = 'password'  # Quotes.toscrape için
        
        form_data.update({
            username_field: test_username,
            password_field: test_password
        })
        
        print(f"📤 Gönderilecek form data:")
        for key, value in form_data.items():
            if 'password' in key.lower():
                print(f"   {key}: {'*' * len(str(value))}")
            else:
                print(f"   {key}: {value}")
        
        # POST request ile login
        headers = {
            'Referer': login_url,
            'Content-Type': 'application/x-www-form-urlencoded'
        }
        
        login_response = session.post(login_url, data=form_data, headers=headers)
        
        print(f"\n📊 Login Response:")
        print(f"   Status: {login_response.status_code}")
        print(f"   Final URL: {login_response.url}")
        print(f"   Cookie sayısı: {len(session.cookies)}")
        
        # ADIM 4: Login başarısını kontrol et
        print("\n4️⃣ Login durumu kontrol ediliyor...")
        
        # Basit kontrol: URL değişikliği ve cookie varlığı
        if login_response.status_code in [200, 302, 303]:
            if len(session.cookies) > 0:
                print("✅ Login başarılı! (Cookie alındı)")
                
                # ADIM 5: Korumalı sayfaya erişim testi
                print("\n5️⃣ Korumalı sayfa erişimi test ediliyor...")
                
                protected_response = session.get(protected_url)
                
                if protected_response.status_code == 200:
                    # Login sonrası sayfa içeriğini kontrol et
                    soup_after_login = BeautifulSoup(protected_response.text, 'html.parser')
                    
                    # Logout linkini ara (login başarısının göstergesi)
                    logout_link = soup_after_login.find('a', href=re.compile(r'logout'))
                    
                    if logout_link:
                        print("✅ Korumalı sayfaya erişim başarılı!")
                        print(f"🚪 Logout linki bulundu: {logout_link.get('href')}")
                        
                        print("\n🎉 TAM LOGIN WORKFLOW BAŞARILI!")
                    else:
                        print("⚠️ Logout linki bulunamadı - Login durumu belirsiz")
                else:
                    print(f"❌ Korumalı sayfaya erişim başarısız: {protected_response.status_code}")
            else:
                print("❌ Login başarısız - Cookie alınamadı")
        else:
            print(f"❌ Login başarısız: {login_response.status_code}")
    else:
        print("❌ Login form'u bulunamadı!")
else:
    print(f"❌ Login sayfası alınamadı: {login_page.status_code}")


1️⃣ Login sayfası getiriliyor...
✅ Login sayfası alındı: 1878 karakter

2️⃣ Form analiz ediliyor...
📋 Login form bulundu!
📝 Form field'ları (4 adet):
   🔸 csrf_token (hidden): 'USswQxjrgBXCaHeGOVqTJZuionNAIfztWFdKYLRhypMlkcbmPvDE'
   🔸 username (text): ''
   🔸 password (password): ''
🔍 CSRF token aranıyor...
✅ CSRF token bulundu! Field: csrf_token
   Token: USswQxjrgBXCaHeGOVqT...
🛡️ CSRF token eklendi: csrf_token

3️⃣ Login bilgileri hazırlanıyor...
📤 Gönderilecek form data:
   csrf_token: USswQxjrgBXCaHeGOVqTJZuionNAIfztWFdKYLRhypMlkcbmPvDE
   username: admin
   password: *****

📊 Login Response:
   Status: 200
   Final URL: http://quotes.toscrape.com/
   Cookie sayısı: 1

4️⃣ Login durumu kontrol ediliyor...
✅ Login başarılı! (Cookie alındı)

5️⃣ Korumalı sayfa erişimi test ediliyor...
✅ Korumalı sayfaya erişim başarılı!
🚪 Logout linki bulundu: /logout

🎉 TAM LOGIN WORKFLOW BAŞARILI!


---

## 🎯 Login Scraping En İyi Pratikleri

### ✅ Yapılması Gerekenler:

1. **Session Kullanın**
   ```python
   session = requests.Session()
   # Tüm istekleri session ile yapın
   ```

2. **Realistic Headers**
   ```python
   headers = {
       'User-Agent': 'Real browser user agent',
       'Referer': 'Previous page URL'
   }
   ```

3. **CSRF Token Kontrolü**
   ```python
   csrf_token = soup.find('input', {'name': 'csrf_token'})['value']
   form_data['csrf_token'] = csrf_token
   ```

4. **Hata Yönetimi**
   ```python
   try:
       response = session.post(url, data=form_data)
   except requests.exceptions.RequestException as e:
       print(f'Hata: {e}')
   ```

5. **Rate Limiting**
   ```python
   time.sleep(1)  # İstekler arası bekleme
   ```

### ❌ Yapılmaması Gerekenler:

- 🚫 **Çok hızlı istek** gönderme
- 🚫 **Gerçek kullanıcı bilgileri** ile test
- 🚫 **Bot olduğunuzu belli etme**
- 🚫 **ToS ihlali** yapma
- 🚫 **Güvenlik önlemlerini bypass** etmeye çalışma

---

## 🎓 Özet: Login Scraping Mastery

Bu notebook'ta **login scraping'in tüm yönlerini** öğrendik:

### ✅ Teknik Beceriler
- **Session management** - Oturum yönetimi
- **Cookie handling** - Çerez yönetimi
- **Form data** gönderme
- **CSRF token** yönetimi
- **Login durumu** kontrolü
- **Hata yönetimi** ve troubleshooting

### ✅ Güvenlik Bilinci
- **Etik kullanım** prensipleri
- **Rate limiting** uygulaması
- **Realistic behavior** simülasyonu
- **Anti-bot önlemlerine** saygı

### 🚀 Sıradaki Seviye
1. **Scrapy ile login** - Framework yaklaşımı
2. **JavaScript rendering** - Modern siteler
3. **2FA handling** - İki faktörlü doğrulama
4. **Proxy rotation** - IP değiştirme

### 💡 Unutmayın
- Login scraping **en zorlu** web scraping konularından biri
- **Çok pratik** gerektirir
- **Etik sınırları** hiç aşmayın
- **API** varsa hep API'yi tercih edin

---

## 📝 Pratik Ödevler

### 🟢 Başlangıç Seviyesi

1. **Session Pratiği**
   - `httpbin.org/cookies` ile session test edin
   - Cookie'lerin nasıl saklandığını gözlemleyin
   - Normal request vs session farkını test edin

2. **Form Analizi**
   - Farklı sitelerin login formlarını inceleyin
   - Hidden field'ları tespit edin
   - CSRF token pattern'lerini öğrenin

### 🟡 Orta Seviye

1. **Quotes.toscrape Login**
   - Bu notebook'taki örneği çalıştırın
   - Login öncesi/sonrası sayfa farklarını analiz edin
   - Logout işlemini de implement edin

2. **Login Status Detection**
   - Farklı başarı göstergeleri test edin
   - Güven skorunu optimize edin
   - False positive/negative durumları minimize edin

### 🔴 İleri Seviye

1. **Multi-Step Authentication**
   - Email verification simülasyonu
   - 2FA token handling
   - Remember me functionality

2. **Error Recovery**
   - Login retry mekanizması
   - Session renewal
   - Captcha detection ve handling

### ⚠️ ETİK KURALLAR

**Bu ödevleri yaparken:**
- ✅ Sadece test siteleri kullanın
- ✅ Rate limiting uygulayın
- ✅ Gerçek kullanıcı bilgileri kullanmayın
- ✅ Robots.txt'ye saygı gösterin
- ❌ Başkalarının hesaplarını hedef almayın
- ❌ Güvenlik önlemlerini bypass etmeye çalışmayın

---

## 🎉 Tebrikler!

**Login scraping'in tüm inceliklerini öğrendiniz!**

Bu notebook'ta edindiğiniz beceriler:
- 🔐 **Session-based authentication**
- 🍪 **Cookie management**
- 🛡️ **CSRF protection handling**
- 📝 **Form data processing**
- 🔍 **Login status detection**
- ⚡ **Error handling & recovery**

### 💪 Artık Hazırsınız

- ✅ Karmaşık login sistemlerini handle edebilirsiniz
- ✅ Session management'ı profesyonel seviyede yapabilirsiniz
- ✅ Güvenlik önlemlerini etik şekilde yönetebilirsiniz

**Happy Scraping! 🕷️**